# Supplementary SX: Proteome-Level Descriptive Comparison Across Four Legume Species

This supplementary workflow performs a descriptive comparison of proteome-level amino-acid composition across *Glycine max*, *Cicer arietinum*, *Lupinus albus*, and *Phaseolus vulgaris*.

Unlike the species-specific notebooks, this comparative workflow imports the four reference-proteome FASTA files directly and applies the same `saa_proteome` processing settings to every species within one notebook. It then derives one proteome-level composition row per species and calculates cross-species descriptive statistics.

The analysis evaluates:

- methionine percentage;
- cysteine percentage;
- total sulfur-containing amino acids (S-AA = Met + Cys);
- glutamic acid percentage as a non-sulfur reference amino acid.

For each metric, the workflow calculates the mean, standard deviation, minimum, maximum, range, and coefficient of variation (CV) across the four analyzed reference proteomes.

> **Primary-analysis setting:** `REMOVE_START_M = True`; therefore, the N-terminal methionine is removed when it occurs as the first residue of the normalized original protein sequence, before optional canonical-residue filtering

The coefficient of variation is calculated as:

$$
\mathrm{CV}(\%) = \frac{\mathrm{SD}}{\mathrm{mean}} \times 100
$$

These statistics are descriptive and do not constitute an inferential hypothesis test.

## 1. Imports and global analysis parameters

The same sequence-processing settings are applied to all four FASTA files to preserve methodological consistency.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from saa_proteome import aa_composition_df, validate_fasta

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda value: f"{value:.6f}")

# Global preprocessing settings
CANONICAL_ONLY = True
REMOVE_START_M = True
VALUE_SCALE = "pct"

# Descriptive standard-deviation convention
DDOF = 0

## 2. Define the reference-proteome FASTA files

Update `PROJECT_ROOT` or the filenames when the notebook is moved. The manifest explicitly links each species label to its FASTA input, making the data sources transparent and reproducible.

In [2]:
# Define the project root as the directory containing this notebook.
PROJECT_ROOT = Path.cwd()

# Folder containing the four reference-proteome FASTA files.
FASTA_DIR = PROJECT_ROOT / "data"

FASTA_FILES = {
    "Glycine max": FASTA_DIR / "UP000008827_3847.fasta",
    "Cicer arietinum": FASTA_DIR / "UP000087171_3827.fasta",
    "Lupinus albus": FASTA_DIR / "UP000447434_3870.fasta",
    "Phaseolus vulgaris": FASTA_DIR / "UP000000226_3885.fasta",
}

df_sources = pd.DataFrame(
    {
        "species": list(FASTA_FILES.keys()),
        "fasta_path": [str(path) for path in FASTA_FILES.values()],
        "file_exists": [path.exists() for path in FASTA_FILES.values()],
    }
)

df_sources

,species,fasta_path,file_exists
0,Glycine max,i:\Other computers\My Laptop\Desktop\Proteome_...,True
1,Cicer arietinum,i:\Other computers\My Laptop\Desktop\Proteome_...,True
2,Lupinus albus,i:\Other computers\My Laptop\Desktop\Proteome_...,True
3,Phaseolus vulgaris,i:\Other computers\My Laptop\Desktop\Proteome_...,True


## 3. Validate the FASTA inputs

Each file is checked for readability, FASTA structure, and parseability before proteome-level analysis.

In [3]:
validation_tables = []

for species, fasta_path in FASTA_FILES.items():
    df_validation = validate_fasta(str(fasta_path)).copy()
    df_validation.insert(0, "species", species)
    validation_tables.append(df_validation)

df_validation_all = pd.concat(validation_tables, ignore_index=True)

df_validation_all

,species,path,readable,looks_like_fasta,parseable,records_found_probe,first_protein_id,error_message
0,Glycine max,i:\Other computers\My Laptop\Desktop\Proteome_...,True,True,True,1,tr|A0A0R0E6H1|A0A0R0E6H1_SOYBN,None
1,Cicer arietinum,i:\Other computers\My Laptop\Desktop\Proteome_...,True,True,True,1,tr|A0A1S2X9S8|A0A1S2X9S8_CICAR,None
2,Lupinus albus,i:\Other computers\My Laptop\Desktop\Proteome_...,True,True,True,1,tr|A0A6A4MVX5|A0A6A4MVX5_LUPAL,None
3,Phaseolus vulgaris,i:\Other computers\My Laptop\Desktop\Proteome_...,True,True,True,1,tr|A0ACC3P112|A0ACC3P112_PHAVU,None


In [4]:
# Stop the workflow if any FASTA input is unavailable or invalid.
required_flags = ["readable", "looks_like_fasta", "parseable"]

invalid_inputs = df_validation_all[
    ~df_validation_all[required_flags].all(axis=1)
]

if not invalid_inputs.empty:
    raise ValueError(
        "One or more FASTA files failed validation. "
        "Review df_validation_all before continuing."
    )

## 4. Derive proteome-level amino-acid composition

Each reference proteome is analyzed independently with `aa_composition_df(level="proteome")`. The resulting rows are then combined for scientific comparison. The core package therefore retains its single-proteome analytical design, while this notebook provides a manuscript-specific multi-species workflow.

In [5]:
def analyze_reference_proteome(species, fasta_path):
    """Return one proteome-level amino-acid composition row for one FASTA file."""
    return aa_composition_df(
        fasta_path=str(fasta_path),
        species=species,
        level="proteome",
        canonical_only=CANONICAL_ONLY,
        remove_start_m=REMOVE_START_M,
        value_scale=VALUE_SCALE,
        extended=True,
    )


proteome_tables = [
    analyze_reference_proteome(species, fasta_path)
    for species, fasta_path in FASTA_FILES.items()
]

df_proteome = pd.concat(proteome_tables, ignore_index=True)

df_proteome

,species,level,total_proteins,canonical_only,remove_start_m,proteins_dropped_empty,total_proteins_used,total_aa_adjusted,aa_value_scale,aa_A,aa_C,aa_D,aa_E,aa_F,aa_G,aa_H,aa_I,aa_K,aa_L,aa_M,aa_N,aa_P,aa_Q,aa_R,aa_S,aa_T,aa_V,aa_W,aa_Y
0,Glycine max,proteome,55852,True,True,0,55852,21638070,pct,6.486341,1.921867,5.223202,6.332455,4.322118,6.352059,2.557747,5.423455,6.221234,9.844885,2.187861,4.781328,4.873831,3.741604,5.102262,8.933694,4.993398,6.542737,1.299067,2.858855
1,Cicer arietinum,proteome,24828,True,True,0,24828,10837683,pct,6.103233,1.883585,5.328713,6.386503,4.386722,6.192246,2.518721,5.745130,6.448823,9.591912,2.199917,5.037590,4.760030,3.711624,4.933804,8.974797,5.083365,6.504296,1.262982,2.946008
2,Lupinus albus,proteome,37884,True,True,0,37884,14006835,pct,6.261065,1.846299,5.265187,6.298204,4.368196,6.322192,2.588693,5.649592,6.293292,9.565587,2.207765,4.949541,4.827515,3.706605,4.947092,9.184823,5.046186,6.506266,1.231335,2.934567
3,Phaseolus vulgaris,proteome,28220,True,True,0,28220,11553063,pct,6.459906,1.920521,5.203797,6.441452,4.371101,6.414810,2.539932,5.340800,6.178664,9.870958,2.170377,4.655536,4.856807,3.661168,5.129930,9.038642,4.993671,6.639581,1.292324,2.820023


## 5. Construct the comparative composition table

Methionine, cysteine, and glutamic acid are obtained directly from the full proteome-level amino-acid composition. Total S-AA is calculated as the sum of methionine and cysteine percentages.

In [6]:
df_comparison = (
    df_proteome[
        [
            "species",
            "total_proteins",
            "total_aa_adjusted",
            "aa_M",
            "aa_C",
            "aa_E",
            "aa_N"
        ]
    ]
    .rename(
        columns={
            "aa_M": "met_pct",
            "aa_C": "cys_pct",
            "aa_E": "glu_pct", # Glutamic acid, as a non-sulfur reference amino acid
        }
    )
    .copy()
)

df_comparison["saa_pct"] = (
    pd.to_numeric(df_comparison["met_pct"], errors="raise")
    + pd.to_numeric(df_comparison["cys_pct"], errors="raise")
)

df_comparison = df_comparison[
    [
        "species",
        "total_proteins",
        "total_aa_adjusted",
        "met_pct",
        "cys_pct",
        "saa_pct",
        "glu_pct"
    ]
]

df_comparison

,species,total_proteins,total_aa_adjusted,met_pct,cys_pct,saa_pct,glu_pct
0,Glycine max,55852,21638070,2.187861,1.921867,4.109729,6.332455
1,Cicer arietinum,24828,10837683,2.199917,1.883585,4.083502,6.386503
2,Lupinus albus,37884,14006835,2.207765,1.846299,4.054064,6.298204
3,Phaseolus vulgaris,28220,11553063,2.170377,1.920521,4.090898,6.441452


## 6. Table 1 verification view

This display reproduces the proteome-level composition fields used in Table 1. Rounding is applied only for presentation. The descriptive calculations below use the unrounded values retained in `df_comparison`.

In [7]:
df_table1_view = df_comparison.copy()

percentage_columns = ["met_pct", "cys_pct", "saa_pct", "glu_pct"]
df_table1_view[percentage_columns] = (
    df_table1_view[percentage_columns].round(2)
)

df_table1_view

,species,total_proteins,total_aa_adjusted,met_pct,cys_pct,saa_pct,glu_pct
0,Glycine max,55852,21638070,2.190000,1.920000,4.110000,6.330000
1,Cicer arietinum,24828,10837683,2.200000,1.880000,4.080000,6.390000
2,Lupinus albus,37884,14006835,2.210000,1.850000,4.050000,6.300000
3,Phaseolus vulgaris,28220,11553063,2.170000,1.920000,4.090000,6.440000


## 7. Descriptive comparison across the four proteomes

The helper function below is local to this supplementary scientific workflow and is not part of the core `saa_proteome` package.

`DDOF = 0` applies the population standard-deviation formula because the calculation describes the complete set of four reference proteomes analyzed in this study.

In [8]:
METRIC_LABELS = {
    "met_pct": "Methionine",
    "cys_pct": "Cysteine",
    "saa_pct": "Total S-AA",
    "glu_pct": "Glutamic acid"
}


def descriptive_comparison(df, metric_labels, ddof=0):
    """Calculate descriptive statistics across proteome-level rows."""
    if not isinstance(df, pd.DataFrame):
        raise TypeError("df must be a pandas DataFrame.")

    if len(df) < 2:
        raise ValueError("At least two proteome-level rows are required.")

    rows = []

    for column, label in metric_labels.items():
        if column not in df.columns:
            raise ValueError(f"Required metric column not found: {column}")

        values = pd.to_numeric(df[column], errors="coerce").dropna()

        if len(values) != len(df):
            raise ValueError(
                f"{label}: missing or non-numeric values were detected."
            )

        mean_value = float(values.mean())
        sd_value = float(values.std(ddof=ddof))
        min_value = float(values.min())
        max_value = float(values.max())
        range_value = max_value - min_value

        cv_value = (
            (sd_value / mean_value) * 100.0
            if not np.isclose(mean_value, 0.0)
            else np.nan
        )

        rows.append(
            {
                "metric": label,
                "n_species": len(values),
                "mean_pct": mean_value,
                "sd_pct": sd_value,
                "minimum_pct": min_value,
                "maximum_pct": max_value,
                "range_percentage_points": range_value,
                "cv_pct": cv_value,
                "ddof": ddof,
            }
        )

    return pd.DataFrame(rows)

In [9]:
df_descriptive = descriptive_comparison(
    df=df_comparison,
    metric_labels=METRIC_LABELS,
    ddof=DDOF,
)

df_descriptive

,metric,n_species,mean_pct,sd_pct,minimum_pct,maximum_pct,range_percentage_points,cv_pct,ddof
0,Methionine,4,2.191480,0.014096,2.170377,2.207765,0.037388,0.643230,0
1,Cysteine,4,1.893068,0.031066,1.846299,1.921867,0.075569,1.641036,0
2,Total S-AA,4,4.084548,0.020030,4.054064,4.109729,0.055665,0.490382,0
3,Glutamic acid,4,6.364653,0.054378,6.298204,6.441452,0.143248,0.854371,0


## 8. Publication-oriented descriptive summary

The table below is rounded for reporting. A lower CV indicates lower relative variation among the four analyzed reference proteomes. These values should not be generalized beyond the species included in this study.

In [10]:
df_descriptive_report = df_descriptive[
    [
        "metric",
        "n_species",
        "mean_pct",
        "sd_pct",
        "minimum_pct",
        "maximum_pct",
        "range_percentage_points",
        "cv_pct",
    ]
].copy()

numeric_report_columns = [
    "mean_pct",
    "sd_pct",
    "minimum_pct",
    "maximum_pct",
    "range_percentage_points",
    "cv_pct",
]

df_descriptive_report[numeric_report_columns] = (
    df_descriptive_report[numeric_report_columns].round(4)
)

df_descriptive_report

,metric,n_species,mean_pct,sd_pct,minimum_pct,maximum_pct,range_percentage_points,cv_pct
0,Methionine,4,2.191500,0.014100,2.170400,2.207800,0.037400,0.643200
1,Cysteine,4,1.893100,0.031100,1.846300,1.921900,0.075600,1.641000
2,Total S-AA,4,4.084500,0.020000,4.054100,4.109700,0.055700,0.490400
3,Glutamic acid,4,6.364700,0.054400,6.298200,6.441500,0.143200,0.854400


## 9. Relative-variability ranking

The metrics are ordered from the lowest to the highest coefficient of variation to support the interpretation of relative compositional stability.

In [11]:
df_cv_ranking = (
    df_descriptive_report
    .sort_values("cv_pct", ascending=True)
    .reset_index(drop=True)
)

df_cv_ranking.insert(
    0,
    "variability_rank",
    range(1, len(df_cv_ranking) + 1),
)

df_cv_ranking

,variability_rank,metric,n_species,mean_pct,sd_pct,minimum_pct,maximum_pct,range_percentage_points,cv_pct
0,1,Total S-AA,4,4.084500,0.020000,4.054100,4.109700,0.055700,0.490400
1,2,Methionine,4,2.191500,0.014100,2.170400,2.207800,0.037400,0.643200
2,3,Glutamic acid,4,6.364700,0.054400,6.298200,6.441500,0.143200,0.854400
3,4,Cysteine,4,1.893100,0.031100,1.846300,1.921900,0.075600,1.641000


## 10. Reproducibility record

This record documents the FASTA inputs, processing settings, and software versions used in the comparative analysis.

In [12]:
import saa_proteome

reproducibility_record = {
    "analysis_scope": "Descriptive comparison across four reference proteomes",
    "input_type": "Reference-proteome FASTA",
    "canonical_only": CANONICAL_ONLY,
    "remove_start_m": REMOVE_START_M,
    "amino_acid_value_scale": VALUE_SCALE,
    "standard_deviation_ddof": DDOF,
    "saa_proteome_version": saa_proteome.__version__,
    "pandas_version": pd.__version__,
    "numpy_version": np.__version__,
    "input_files": {
        species: str(path)
        for species, path in FASTA_FILES.items()
    },
}

reproducibility_record

{'analysis_scope': 'Descriptive comparison across four reference proteomes',
 'input_type': 'Reference-proteome FASTA',
 'canonical_only': True,
 'remove_start_m': True,
 'amino_acid_value_scale': 'pct',
 'standard_deviation_ddof': 0,
 'saa_proteome_version': '0.4.0',
 'pandas_version': '2.3.1',
 'numpy_version': '2.2.1',
 'input_files': {'Glycine max': 'i:\\Other computers\\My Laptop\\Desktop\\Proteome_analysis_S_AA\\data\\UP000008827_3847.fasta',
  'Cicer arietinum': 'i:\\Other computers\\My Laptop\\Desktop\\Proteome_analysis_S_AA\\data\\UP000087171_3827.fasta',
  'Lupinus albus': 'i:\\Other computers\\My Laptop\\Desktop\\Proteome_analysis_S_AA\\data\\UP000447434_3870.fasta',
  'Phaseolus vulgaris': 'i:\\Other computers\\My Laptop\\Desktop\\Proteome_analysis_S_AA\\data\\UP000000226_3885.fasta'}}

## Interpretation guidance

> Each reference proteome was processed independently using identical `saa_proteome` settings within a separate comparative workflow. Proteome-level methionine, cysteine, total sulfur-containing amino-acid, and glutamic-acid percentages were derived directly from the four FASTA inputs. Cross-species variability was then summarized descriptively using the mean, population standard deviation, range, and coefficient of variation, with the coefficient of variation expressed as 100 × SD/mean.